# 07 - SAMap inputs and the gene-map sanity check

* SAMap inputs: mouse = annotation of record without 'unresolved', at most 4,000 cells per type (53,370); human = HLiCA v1.0,
  at most 1,200 per author_cell_type (50,400); var_names renamed to the SAMap gene names of the MMseqs2 maps (heavy notebook 11;
  record linked).
* Gene maps vs MGI curated 1:1 orthologs (Ensembl BioMart and HGNC HCOP could not be downloaded from the VM): result of record
  90.1 % (human->mouse) and 91.1 % (mouse->human) top hit, 97.6 % among the hits.

**Runtime:** ~10 min
**Envs used (each step runs as a subprocess in its own env):** samap
**Kernel:** `masld_atlas` (orchestration only).

Outputs are written under `/mnt/masld_vault/cross_species_repro/` (mirror of `/mnt/masld_vault/cross_species/`, the outputs
of record). Inputs this notebook does not rebuild are linked from the record; see `repro_setup.py` for the safety rules.

In [ ]:
%run -i /home/ubuntu/liver_cross_species/notebooks/repro_setup.py
print("RECORD", RECORD); print("REPRO ", REPRO); print("WORK  ", WORK)

In [ ]:
link("guilliams2022/13_annotation_of_record", "samap_maps/liver_hsmm/refs/mappings", "samap_maps/liver_hsmm/maps_mmseqs")
fresh("samap_run/input")

### SAMap prep

In [ ]:
%%writefile /home/ubuntu/liver_cross_species/notebooks/work/01_prep.py
"""SAMap prep (2026-11-12): HLiCA v1.0 x Guilliams 2022 mouse atlas (annotation of record, 13b).
Subsample stratified by cell type (seed 0): mouse cap 4,000 per annotation_aor ('unresolved' excluded); human cap chosen so the
total is ~50k over the 48 author_cell_type labels. X = counts (mouse: clean counts = decontX (1,300) for 13 in vivo samples, raw
otherwise; human: raw.X UMI). var_names -> SAMap names (refs/mappings/{hs,mm}.gene_names.tsv, Symbol or Symbol__EnsemblID);
genes absent from the tables dropped. Outputs /mnt/masld_vault/cross_species_repro/samap_run/input/{hs,mm}_samap_ready.h5ad + subsample tables."""
import anndata as ad, numpy as np, pandas as pd, scipy.sparse as sp
MAPS = "/mnt/masld_vault/cross_species_repro/samap_maps/liver_hsmm/refs/mappings"
O = "/mnt/masld_vault/cross_species_repro/samap_run/input"
rng = np.random.default_rng(0)
def strat(lab, cap):
    idx = []
    for t, ii in pd.Series(np.arange(len(lab))).groupby(lab.values).groups.items():
        ii = np.asarray(ii); idx.append(ii if len(ii) <= cap else rng.choice(ii, cap, replace=False))
    return np.sort(np.concatenate(idx))
def rename(a, tab):
    g = pd.read_csv(tab, sep="\t").set_index("ensembl_gene_id").samap_gene
    keep = a.var_names.isin(g.index); a = a[:, keep].copy(); a.var["ensembl_gene_id"] = a.var_names.astype(str)
    a.var_names = g.loc[a.var_names].astype(str).values; assert a.var_names.is_unique; a.var.index.name = "samap_gene"
    print("genes kept", keep.sum(), "of", len(keep), flush=True); return a
# mouse
m = ad.read_h5ad("/mnt/masld_vault/cross_species_repro/guilliams2022/13_annotation_of_record/guilliams2022_fig1b_aor.h5ad")
m = m[m.obs.annotation_aor.astype(str) != "unresolved"]
ix = strat(m.obs.annotation_aor.astype(str), 4000)
mm = ad.AnnData(X=sp.csr_matrix(m.X[ix]).astype(np.float32), obs=m.obs.iloc[ix][["sample", "suspension", "digestion", "sort_gate", "annotation_aor", "aor_flag"]].copy(),
                var=m.var[["symbol"]].copy())
mm.obs["annotation_aor"] = mm.obs.annotation_aor.astype(str).astype("category"); mm.obsm["X_umap_mouse"] = m.obsm["X_umap"][ix]
del m
mm = rename(mm, f"{MAPS}/mm.gene_names.tsv")
print(mm.obs.annotation_aor.value_counts().to_string(), "\nmouse", mm.shape, flush=True)
mm.obs.annotation_aor.value_counts().to_csv(f"{O}/mm_subsample_counts.tsv", sep="\t")
mm.write_h5ad(f"{O}/mm_samap_ready.h5ad")
# human
h = ad.read_h5ad("/mnt/masld_vault/geo/cxg-059202e1-1f1b-483f-9151-f3a25a380c39/HLiCA_human_liver_all_cells.h5ad", backed="r")
lab = h.obs.author_cell_type.astype(str); vc = lab.value_counts()
cap = next(c for c in range(200, 20000, 50) if np.minimum(vc.values, c).sum() >= 50000)
print("human cap per type", cap, flush=True)
ix = strat(lab, cap)
assert (h.raw.var_names == h.var_names).all()
X = h.raw.X[ix] if not hasattr(h.raw.X, "shape") or True else None
X = sp.csr_matrix(X).astype(np.float32)
hs = ad.AnnData(X=X, obs=h.obs.iloc[ix][["author_cell_type", "cell_type", "suspension_type", "STUDY", "donor_id", "assay", "condition"]].copy(),
                var=h.var[["feature_name", "feature_biotype"]].copy())
hs.obsm["X_umap_hlica"] = np.asarray(h.obsm["X_umap"][ix])
for c in hs.obs.columns:
    if hs.obs[c].dtype.name == "category": hs.obs[c] = hs.obs[c].cat.remove_unused_categories()
hs = rename(hs, f"{MAPS}/hs.gene_names.tsv")
print(hs.obs.author_cell_type.value_counts().to_string(), "\nhuman", hs.shape, flush=True)
hs.obs.author_cell_type.value_counts().to_csv(f"{O}/hs_subsample_counts.tsv", sep="\t")
hs.write_h5ad(f"{O}/hs_samap_ready.h5ad")
print("done", flush=True)

In [ ]:
run("01_prep.py", "samap")

In [ ]:
compare_tsv("samap_run/input/mm_subsample_counts.tsv")
compare_tsv("samap_run/input/hs_subsample_counts.tsv")

In [ ]:
fresh("samap_maps/liver_hsmm/refs/orthologs")
link("samap_maps/liver_hsmm/refs/orthologs/HOM_MouseHumanSequence.rpt")

### Ortholog check

In [ ]:
%%writefile /home/ubuntu/liver_cross_species/notebooks/work/06_ortholog_check.py
"""Step 6 - sanity check of the MMseqs2 maps against MGI curated 1:1 orthologs (HOM_MouseHumanSequence.rpt, fetched 2026-11-12;
Ensembl BioMart and HCOP downloads failed from the VM). 1:1 = homology class with exactly one mouse and one human gene.
Restricted to human protein-coding genes (HLiCA var feature_type). Per direction: of the 1:1 pairs whose two genes are in our
gene tables (matched by symbol = samap_gene), the fraction where the ortholog is (a) the top hit by bits, (b) among the hits at all.
Output refs/orthologs/ortholog_check.tsv + printed summary."""
import anndata as ad, pandas as pd
R = "/mnt/masld_vault/cross_species_repro/samap_maps/liver_hsmm"
cols = ["q", "t", "pident", "alnlen", "mm", "gap", "qs", "qe", "ts", "te", "evalue", "bits"]
mgi = pd.read_csv(f"{R}/refs/orthologs/HOM_MouseHumanSequence.rpt", sep="\t", dtype=str)
g = mgi.groupby("DB Class Key")
one = g.filter(lambda x: (x["NCBI Taxon ID"] == "10090").sum() == 1 and (x["NCBI Taxon ID"] == "9606").sum() == 1)
p = one.pivot_table(index="DB Class Key", columns="NCBI Taxon ID", values="Symbol", aggfunc="first").rename(columns={"9606": "hs", "10090": "mm"})
hv = ad.read_h5ad("/mnt/masld_vault/geo/cxg-059202e1-1f1b-483f-9151-f3a25a380c39/HLiCA_human_liver_all_cells.h5ad", backed="r").var
pc = set(hv.feature_name[hv.feature_type.astype(str) == "protein_coding"].astype(str))
hsn = set(pd.read_csv(f"{R}/refs/mappings/hs.gene_names.tsv", sep="\t").samap_gene); mmn = set(pd.read_csv(f"{R}/refs/mappings/mm.gene_names.tsv", sep="\t").samap_gene)
p = p[p.hs.isin(pc) & p.hs.isin(hsn) & p.mm.isin(mmn)]
print("MGI 1:1 pairs, human protein-coding, both genes in our tables:", len(p), flush=True)
out = []
for d, qcol, tcol in (("hs_to_mm", "hs", "mm"), ("mm_to_hs", "mm", "hs")):
    m = pd.read_csv(f"{R}/maps_mmseqs/hsmm/{d}.txt", sep="\t", header=None, names=cols, usecols=["q", "t", "bits", "pident"])
    m = m[m.q.isin(set(p[qcol]))]
    top = m.sort_values("bits", ascending=False).drop_duplicates("q").set_index("q").t
    pairs = set(zip(m.q, m.t))
    exp = p.set_index(qcol)[tcol]
    hit = exp.index.isin(top.index)
    r = {"direction": d, "n_pairs": len(exp), "query_has_hits": hit.mean(),
         "ortholog_top_hit": (top.reindex(exp.index) == exp).mean(),
         "ortholog_among_hits": pd.Series([(q, t) in pairs for q, t in exp.items()]).mean()}
    out.append(r); print(r, flush=True)
pd.DataFrame(out).to_csv(f"{R}/refs/orthologs/ortholog_check.tsv", sep="\t", index=False, float_format="%.4f")

In [ ]:
run("06_ortholog_check.py", "samap")

In [ ]:
compare_tsv("samap_maps/liver_hsmm/refs/orthologs/ortholog_check.tsv", index_col=None)